### 工作進度  
* 【置頂】**筆記內容架構**與**量化技術分析系統**相關資訊請參閱[260531筆記.ipynb](https://github.com/yilintung/StockInvestmentNotebook/blob/main/260531%E7%AD%86%E8%A8%98.ipynb)之「工作進度」。  
* 周末重新建立量化技術分析資料庫，相關技術資訊請參照[FinMind API](https://finmind.github.io/tutor/TaiwanMarket/Technical/)。  

In [1]:
import os
import sys
import pandas as pd
import datetime
import sqlite3
import requests
import time

from dotenv import load_dotenv, find_dotenv
from FinMind.data import DataLoader

In [2]:
# python 取得時間範圍內日期列表
# 來源：https://www.cnblogs.com/xiao-xue-di/p/11900649.html

def date_range(beginDate, endDate):
    dates = []
    dt = datetime.datetime.strptime(beginDate, "%Y-%m-%d")
    date = beginDate[:]
    while date <= endDate:
        dates.append(date)
        dt = dt + datetime.timedelta(1)
        date = dt.strftime("%Y-%m-%d")
    return dates

In [3]:
# 取得範圍日期列表(2019年12月30日至2025年5月6日)
price_date_list = date_range('2019-12-30', '2025-05-06')

In [4]:
# 設定FinMind API
load_dotenv(find_dotenv())
token = os.environ.get('FINMIND_TOKEN')
api = DataLoader()
api.login_by_token(api_token=token)

2026-06-13 15:40:53.984 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-06-13 15:40:54.037 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success


True

In [5]:
#API 使用次數
def api_usage() :
    url = "https://api.web.finmindtrade.com/v2/user_info"
    payload = {
        "token": token,
    }
    resp = requests.get(url, params=payload)
    return resp.json()["user_count"],resp.json()["api_request_limit"]

In [6]:
print(api_usage())

(0, 6000)


In [7]:
# 連線資料庫
conn = sqlite3.connect('data/stock.db')
cursor = conn.cursor()

In [8]:
# 啟用外鍵支持
conn.execute('PRAGMA foreign_keys = ON;')

res = conn.execute('PRAGMA foreign_keys;')
res.fetchone()

(1,)

In [9]:
# 設定資料庫結構
cursor.execute('''
CREATE TABLE IF NOT EXISTS StockInfo
(
    StockID TEXT NOT NULL PRIMARY KEY ,
    StockName TEXT,
    IndustryCategory TEXT,
    Type TEXT
);
''')
cursor.execute('''
CREATE TABLE IF NOT EXISTS DailyPrice
(
    SerialNo INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
    StockID TEXT NOT NULL,
    Date TEXT,
    Open REAL,
    High REAL,
    Low REAL,
    Close REAL,
    Volume INTEGER,
    Value INTEGER,
    
    FOREIGN KEY (StockID) REFERENCES StockInfo (StockID)
);
''')
cursor.execute('''
CREATE TABLE IF NOT EXISTS WeeklyPrice
(
    SerialNo INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
    StockID TEXT NOT NULL,
    Date TEXT,
    Open REAL,
    High REAL,
    Low REAL,
    Close REAL,
    Volume INTEGER,
    Value INTEGER,
    
    FOREIGN KEY (StockID) REFERENCES StockInfo (StockID)
);
''')
cursor.execute('''
CREATE TABLE IF NOT EXISTS MonthlyPrice
(
    SerialNo INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
    StockID TEXT NOT NULL,
    Date TEXT,
    Open REAL,
    High REAL,
    Low REAL,
    Close REAL,
    Volume INTEGER,
    Value INTEGER,
    
    FOREIGN KEY (StockID) REFERENCES StockInfo (StockID)
);
''')
conn.commit()

In [10]:
# 台股總覽 TaiwanStockInfo #
df = api.taiwan_stock_info()
# display(df)
df_stock_info = df.drop(columns=['date'])
df_stock_info = df_stock_info.rename(columns={'stock_id':'StockID','stock_name':'StockName','industry_category':'IndustryCategory','type':'Type'})
df_stock_info.drop_duplicates(subset=['StockID'], keep='first', inplace=True)
display(df_stock_info)
for idx in range(df_stock_info.shape[0]) :
    info = df_stock_info.iloc[idx]
    conn.execute("INSERT INTO StockInfo(StockID,StockName,IndustryCategory,Type) VALUES(?,?,?,?)", (info['StockID'],info['StockName'],info['IndustryCategory'],info['Type']))

2026-06-13 15:40:54.328 | INFO     | FinMind.data.finmind_api:get_data:164 - download TaiwanStockInfo, data_id: 


,IndustryCategory,StockID,StockName,Type
0,電子零組件業,5481,新華,tpex
1,光電業,3629,地心引力,tpex
2,文化創意業,3687,歐買尬,tpex
3,電腦及週邊設備業,5450,寶聯通,tpex
4,其他電子類,6238,勝麗,tpex
...,...,...,...,...
4135,Index,BiotechnologyMedicalCare,生技醫療類指數,twse
4136,Index,Automobile,汽車類指數,twse
4137,大盤,TPEx,櫃買指數,tpex
4138,Index,Food,食品類指數,twse


In [11]:
for price_date in price_date_list :
    # API 使用次數 #
    user_count,api_request_limit = api_usage()
    if user_count > (api_request_limit - 100) :
        print('抓取資料速度過快（user_count＝ {} ，api_request_limit ＝ {}），等三十分鐘後再行抓取'.format(user_count,api_request_limit))
        time.sleep(30*60)
        
    # 股價日成交資訊 TaiwanStockPrice：一次拿特定日期，所有資料(只限 backer、sponsor 使用) #
    while True :
        try :
            df = api.taiwan_stock_daily(start_date=price_date,)
        except Exception as e:
            print('日K：日期{}發生錯誤{}，重試'.format(price_date,e))
            continue
        break
    if df.empty is not True :
        print('日K：{}'.format(price_date))
        df_daily_price = df.drop(columns=['spread','Trading_turnover'])
        df_daily_price = df_daily_price.rename(columns={'date':'Date','stock_id':'StockID','Trading_Volume':'Volume','Trading_money':'Value','open':'Open','max':'High','min':'Low','close':'Close'})
        # 保存格式：日期、股票代碼、開盤價、最高價、最低價、收盤價、成交量與成交值
        df_daily_price = df_daily_price[['Date', 'StockID', 'Open', 'High', 'Low', 'Close', 'Volume', 'Value']]
        # 排除掉非TaiwanStockInfo內的股票
        df_daily_price = df_daily_price[df_daily_price['StockID'].isin(df_stock_info['StockID'].to_list())]
        df_daily_price.to_sql('DailyPrice', conn, if_exists='append', index=False)
    else :    
        time.sleep(1)
    
    # 台股週 K 資料表 TaiwanStockWeekPrice (只限 backer、sponsor 會員使用) ： 一次拿特定日期，所有資料(只限 backer、sponsor 使用) #
    url = "https://api.finmindtrade.com/api/v4/data"
    parameter = {
        "dataset": "TaiwanStockWeekPrice",
        "start_date": price_date,
        "token": token,
    }
    while True :
        resp = requests.get(url, params=parameter)
        if resp.status_code == 200 :
            break
        else :
            print('周K：日期{}發生錯誤，回應狀態碼 ＝ {}'.format(price_date,resp.status_code))
            if resp.status_code == 402 :
                print('周K：API 用量超出上限，等十分鐘後重試')
                time.sleep(10*60)
    data = resp.json()
    df_weekly_price = pd.DataFrame(data["data"])
    if df_weekly_price.empty is not True :
        print('周K：{}'.format(price_date))
        df_weekly_price = df_weekly_price.drop(columns=['yweek','spread','trading_turnover'])
        df_weekly_price = df_weekly_price.rename(columns={'date':'Date','stock_id':'StockID','trading_volume':'Volume','trading_money':'Value','open':'Open','max':'High','min':'Low','close':'Close'})
        # 保存格式：日期、股票代碼、開盤價、最高價、最低價、收盤價、成交量與成交值
        df_weekly_price = df_weekly_price[['Date', 'StockID', 'Open', 'High', 'Low', 'Close', 'Volume', 'Value']]
        # 排除掉非TaiwanStockInfo內的股票
        df_weekly_price = df_weekly_price[df_weekly_price['StockID'].isin(df_stock_info['StockID'].to_list())]
        df_weekly_price.to_sql('WeeklyPrice', conn, if_exists='append', index=False)
    else :    
        time.sleep(1)
    
    # 台股月 K 資料表 TaiwanStockMonthPrice (只限 backer、sponsor 會員使用) ： 一次拿特定日期，所有資料(只限 backer、sponsor 使用) #
    url = "https://api.finmindtrade.com/api/v4/data"
    parameter = {
        "dataset": "TaiwanStockMonthPrice",
        "start_date": price_date,
        "token": token, 
    }
    while True :
        resp = requests.get(url, params=parameter)
        if resp.status_code == 200 :
            break
        else :
            print('月K：日期{}發生錯誤，回應狀態碼 ＝ {}'.format(price_date,resp.status_code))
            if resp.status_code == 402 :
                print('月K：API 用量超出上限，等十分鐘後重試')
                time.sleep(10*60)
    data = resp.json()
    df_monthly_price = pd.DataFrame(data["data"])
    if df_monthly_price.empty is not True :
        print('月K：{}'.format(price_date))
        df_monthly_price = df_monthly_price.drop(columns=['ymonth','spread','trading_turnover'])
        df_monthly_price = df_monthly_price.rename(columns={'date':'Date','stock_id':'StockID','trading_volume':'Volume','trading_money':'Value','open':'Open','max':'High','min':'Low','close':'Close'})
        # 保存格式：日期、股票代碼、開盤價、最高價、最低價、收盤價、成交量與成交值
        df_monthly_price = df_monthly_price[['Date', 'StockID', 'Open', 'High', 'Low', 'Close', 'Volume', 'Value']]
        # 排除掉非TaiwanStockInfo內的股票
        df_monthly_price = df_monthly_price[df_monthly_price['StockID'].isin(df_stock_info['StockID'].to_list())]
        df_monthly_price.to_sql('MonthlyPrice', conn, if_exists='append', index=False)
    else :    
        time.sleep(1)

2026-06-13 15:40:55.047 | INFO     | FinMind.data.finmind_api:get_data:164 - download TaiwanStockPrice, data_id: 


日K：2019-12-30
周K：2019-12-30


2026-06-13 15:40:58.522 | INFO     | FinMind.data.finmind_api:get_data:164 - download TaiwanStockPrice, data_id: 


日K：2019-12-31


2026-06-13 15:41:02.041 | INFO     | FinMind.data.finmind_api:get_data:164 - download TaiwanStockPrice, data_id: 
2026-06-13 15:41:05.346 | INFO     | FinMind.data.finmind_api:get_data:164 - download TaiwanStockPrice, data_id: 


...

日K：2025-05-02


2026-06-13 17:48:47.628 | INFO     | FinMind.data.finmind_api:get_data:164 - download TaiwanStockPrice, data_id: 
2026-06-13 17:48:51.232 | INFO     | FinMind.data.finmind_api:get_data:164 - download TaiwanStockPrice, data_id: 
2026-06-13 17:48:54.827 | INFO     | FinMind.data.finmind_api:get_data:164 - download TaiwanStockPrice, data_id: 


日K：2025-05-05
周K：2025-05-05


2026-06-13 17:48:58.909 | INFO     | FinMind.data.finmind_api:get_data:164 - download TaiwanStockPrice, data_id: 


日K：2025-05-06


In [12]:
# 修改並關閉資料庫
conn.commit()
conn.close()